
# Driver Drowsiness Detection using CNN — Clean Notebook

This notebook is a cleaned and corrected version with a stable model, stratified splits, and proper output/loss pairing.



## 1. Setup & Imports


In [ ]:

import os, sys, math, random, json, time
import numpy as np
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)
np.random.seed(42)
tf.random.set_seed(42)



## 2. Parameters
Adjust these as needed.


In [ ]:

# Image settings
image_height = 64
image_width  = 64
image_channels = 1  # use 1 for grayscale, 3 for RGB

# Training settings
batch_size = 32
epochs     = 30
learning_rate = 1e-3

# Paths — OPTION A: point to a single root directory with subfolders per class
# e.g. data_root/
#          ├── train/
#          │     ├── drowsy/
#          │     └── alert/
#          ├── val/
#          │     ├── drowsy/
#          │     └── alert/
#          └── test/
#                ├── drowsy/
#                └── alert/
data_root = "PATH/TO/DATA_ROOT"  # <-- TODO: set this

train_dir = os.path.join(data_root, "train")
val_dir   = os.path.join(data_root, "val")
test_dir  = os.path.join(data_root, "test")

# If you only have a single folder and want to split automatically, set single_dir and use OPTION B below.
single_dir = "PATH/TO/SINGLE_DATASET_DIR"  # <-- leave as-is if using OPTION A
use_option_b_split = False  # set True to use single folder + automatic split



## 3. Data Pipeline

### Option A (recommended): Pre-split folders (train/val/test)


In [ ]:

if not use_option_b_split:
    assert os.path.isdir(train_dir) and os.path.isdir(val_dir), #        "Train/Val directories not found. Either set them correctly or use OPTION B."

    color_mode = 'grayscale' if image_channels == 1 else 'rgb'

    train_gen = ImageDataGenerator(rescale=1/255.0,
                                   rotation_range=10,
                                   width_shift_range=0.05,
                                   height_shift_range=0.05,
                                   zoom_range=0.05,
                                   horizontal_flip=True)
    val_gen   = ImageDataGenerator(rescale=1/255.0)
    test_gen  = ImageDataGenerator(rescale=1/255.0)

    train_ds = train_gen.flow_from_directory(
        train_dir,
        target_size=(image_height, image_width),
        color_mode=color_mode,
        class_mode='categorical',  # 2 classes -> one-hot labels
        batch_size=batch_size,
        shuffle=True
    )

    val_ds = val_gen.flow_from_directory(
        val_dir,
        target_size=(image_height, image_width),
        color_mode=color_mode,
        class_mode='categorical',
        batch_size=batch_size,
        shuffle=False
    )

    test_ds = None
    if os.path.isdir(test_dir):
        test_ds = test_gen.flow_from_directory(
            test_dir,
            target_size=(image_height, image_width),
            color_mode=color_mode,
            class_mode='categorical',
            batch_size=batch_size,
            shuffle=False
        )

    num_classes = len(train_ds.class_indices)
    print("Detected classes:", train_ds.class_indices)



### Option B: Single folder split (set `use_option_b_split=True`)


In [ ]:

if use_option_b_split:
    assert os.path.isdir(single_dir), "Single dataset directory not found."
    color_mode = 'grayscale' if image_channels == 1 else 'rgb'

    datagen = ImageDataGenerator(rescale=1/255.0, validation_split=0.2,
                                 rotation_range=10,
                                 width_shift_range=0.05,
                                 height_shift_range=0.05,
                                 zoom_range=0.05,
                                 horizontal_flip=True)

    train_ds = datagen.flow_from_directory(
        single_dir,
        target_size=(image_height, image_width),
        color_mode=color_mode,
        class_mode='categorical',
        subset='training',
        batch_size=batch_size,
        shuffle=True,
        seed=42
    )
    val_ds = datagen.flow_from_directory(
        single_dir,
        target_size=(image_height, image_width),
        color_mode=color_mode,
        class_mode='categorical',
        subset='validation',
        batch_size=batch_size,
        shuffle=False,
        seed=42
    )
    test_ds = None
    num_classes = len(train_ds.class_indices)
    print("Detected classes:", train_ds.class_indices)



## 4. Model (Fixed Output/Loss Pairing)
- Uses **2+ Conv blocks**, then dense layers with dropout.
- Final layer uses **`softmax`** with **`num_classes`** outputs.
- Loss is **`categorical_crossentropy`**.


In [ ]:

def build_model(input_shape=(64,64,1), num_classes=2, lr=1e-3):
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        Conv2D(32, (3,3), activation='relu'),
        MaxPooling2D(pool_size=(2,2)),

        Conv2D(64, (3,3), activation='relu'),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(pool_size=(2,2)),

        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),

        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model


In [ ]:

# Infer input shape from settings
input_shape = (image_height, image_width, image_channels)
model = build_model(input_shape=input_shape, num_classes=num_classes, lr=learning_rate)
model.summary()



## 5. Train with Callbacks


In [ ]:

callbacks = [
    EarlyStopping(patience=8, monitor='val_loss', restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1),
    ModelCheckpoint('best_drowsiness_cnn.h5', monitor='val_loss', save_best_only=True, verbose=1),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=callbacks,
    verbose=1
)



## 6. Evaluate


In [ ]:

val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f"Validation accuracy: {val_acc:.4f}")

if 'test_ds' in globals() and test_ds is not None:
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    print(f"Test accuracy: {test_acc:.4f}")



## 7. Classification Report & Confusion Matrix (optional)


In [ ]:

def evaluate_predictions(ds):
    y_true = ds.classes
    y_prob = model.predict(ds, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    labels = list(ds.class_indices.keys())
    print("Labels:", labels)
    print(classification_report(y_true, y_pred, target_names=labels))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

if 'val_ds' in globals() and val_ds is not None:
    print("Validation set:")
    evaluate_predictions(val_ds)

if 'test_ds' in globals() and test_ds is not None:
    print("\nTest set:")
    evaluate_predictions(test_ds)



## 8. Save / Load


In [ ]:

# Save final model (SavedModel format)
model.save("driver_drowsiness_cnn_savedmodel")

# How to load later:
# loaded = tf.keras.models.load_model("driver_drowsiness_cnn_savedmodel")



---

## Appendix: If you already have `images` and `labels` as NumPy arrays
Uncomment and adapt the code below to use arrays directly.


In [ ]:

# %% OPTIONAL: Use NumPy arrays
# images: shape (N, H, W[, C]), labels: integer class ids in [0..num_classes-1]
# images = ...
# labels = ...

# images = images.astype('float32') / 255.0
# X_train, X_temp, y_train, y_temp = train_test_split(
#     images, labels, test_size=0.3, stratify=labels, random_state=42
# )
# X_val, X_test, y_val, y_test = train_test_split(
#     X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
# )
# num_classes = len(np.unique(labels))

# if images.ndim == 3:
#     X_train = np.expand_dims(X_train, -1)
#     X_val   = np.expand_dims(X_val, -1)
#     X_test  = np.expand_dims(X_test, -1)

# y_train = to_categorical(y_train, num_classes=num_classes)
# y_val   = to_categorical(y_val,   num_classes=num_classes)
# y_test  = to_categorical(y_test,  num_classes=num_classes)

# model = build_model(input_shape=X_train.shape[1:], num_classes=num_classes, lr=learning_rate)
# model.summary()

# callbacks = [
#     EarlyStopping(patience=8, monitor='val_loss', restore_best_weights=True),
#     ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1),
#     ModelCheckpoint('best_drowsiness_cnn.h5', monitor='val_loss', save_best_only=True, verbose=1),
# ]

# history = model.fit(
#     X_train, y_train,
#     validation_data=(X_val, y_val),
#     epochs=epochs,
#     batch_size=batch_size,
#     callbacks=callbacks,
#     verbose=1
# )

# val_loss, val_acc = model.evaluate(X_val, verbose=0)
# print(f"Validation accuracy: {val_acc:.4f}")
# test_loss, test_acc = model.evaluate(X_test, verbose=0)
# print(f"Test accuracy: {test_acc:.4f}")
